# Notebook for running basic sanity checks on the DFT results

In [1]:
import os
import re
import copy
import numpy as np
import glob
import sys

import rmgpy.kinetics
import rmgpy.tools.uncertainty


DFT_DIR = os.path.join(os.environ['AUTOSCIENCE_REPO'], 'dft')
sys.path.append(DFT_DIR)
import autotst_wrapper

sys.path.append(os.environ['DATABASE_DIR'])
import database_fun


import matplotlib.pyplot as plt
%matplotlib inline

/home/harris.se/rmg/RMG-Py/rmgpy/rmg/reactors.py:53: RuntimeWarning: Unable to import Julia dependencies, original error: [Errno 2] No such file or directory: 'julia': 'julia'
  warnings.warn("Unable to import Julia dependencies, original error: " + str(e), RuntimeWarning)


Loading DFT database from /work/westgroup/harris.se/autoscience/reaction_calculator/database


In [2]:
def get_i_thing(thing, thing_list):
    for i in range(len(thing_list)):
        if thing.is_isomorphic(thing_list[i]):
            return i
    return -1

def get_reverse_reaction(reaction):
    assert reaction.kinetics is not None
    rev_reaction = copy.deepcopy(reaction)
    tmp_reactants = rev_reaction.reactants
    rev_reaction.reactants = rev_reaction.products
    rev_reaction.products = tmp_reactants
    rev_reaction.kinetics = reaction.generate_reverse_rate_coefficient()
    return rev_reaction

## Load all of the calculations

In [40]:
# load the calculations
# gather all of the calculations
thermo_libs = glob.glob(os.path.join(DFT_DIR, 'thermo', 'species*', 'arkane', 'RMG_libraries'))

# Load the Arkane thermo
entries = []
for i, lib_path in enumerate(thermo_libs):
    matches = re.search('species_([0-9]{4})', lib_path)
    species_index = int(matches[1])
    ark_thermo_database = rmgpy.data.thermo.ThermoDatabase()
    ark_thermo_database.load_libraries(
        lib_path,
    )

    for key in ark_thermo_database.libraries['thermo'].entries.keys():
        entry = ark_thermo_database.libraries['thermo'].entries[key]
        entry.index = species_index
        entry.label = entry.item.smiles
        entries.append(entry)
print(f'{len(entries)} thermo entries')

sp_entries = [e.item for e in entries]


# first, get valid kinetics from old workflow
kinetics_libs = glob.glob(os.path.join(DFT_DIR, 'kinetics', 'reaction*', 'arkane', 'RMG_libraries'))

# Load the Arkane kinetics
k_entries = []
for i, lib_path in enumerate(kinetics_libs):
    
    matches = re.search('reaction_([0-9]{4,6})', lib_path)
    reaction_index = int(matches[1])
    
    ark_kinetics_database = rmgpy.data.kinetics.KineticsDatabase()
    ark_kinetics_database.load_libraries(lib_path)
    
    
    
    
    # TODO fix bug related to load_libraries not getting the actual name
    for key in ark_kinetics_database.libraries[''].entries.keys():
        entry = ark_kinetics_database.libraries[''].entries[key]
        
        
        # check isomorphism with include_list
        idx = database_fun.get_unique_reaction_index(ark_kinetics_database.libraries[''].entries[key].item)

        entry.index = reaction_index
        k_entries.append(entry)
#         print(f'Adding\t{entry.index}\t{entry}')
print(f'{len(k_entries)} kinetics entries')

rxn_entries = [e.item for e in k_entries]

108 thermo entries
47 kinetics entries


## 1. Compare results to other methods

### 1.1 Do single point energies with AE corrections differ by more than 3.35 kcal/mol between m06-2x and dlpno-CCSD(T)?

In [4]:
np.sqrt(np.float_power(3.0, 2) + np.float_power(1.5, 2))  # uncertainties of m06-2x and dlpno-CCSD(T) added in quadrature

3.3541019662496847

### 1.2. Do any of the species thermodynamics differ by more than the GAV + dlpno-CCSD(T) uncertainty?

In [5]:
# Need to load a thermo database to get GAV


database = rmgpy.data.rmg.RMGDatabase()

thermo_libraries = [
    'BurkeH2O2',
    'primaryThermoLibrary',
]

database.load(
    path = rmgpy.settings['database.directory'],
    thermo_libraries = thermo_libraries,
    transport_libraries = [],
    reaction_libraries = [],
    seed_mechanisms = [],#['BurkeH2O2inN2','ERC-FoundationFuelv0.9'],
    kinetics_families = 'default',
    kinetics_depositories = ['training'],
    #frequenciesLibraries = self.statmechLibraries,
    depository = False, # Don't bother loading the depository information, as we don't use it
)



In [36]:
# Get all species GAV uncertainties
GAV_species = [rmgpy.species.Species(molecule=[entry.item]) for entry in entries]
for i in range(len(GAV_species)):
    GAV_species[i].thermo = database.thermo.get_thermo_data(GAV_species[i])

uncertainty = rmgpy.tools.uncertainty.Uncertainty(species_list=GAV_species, reaction_list=[])
uncertainty.database = database

uncertainty.extract_sources_from_model()
uncertainty.assign_parameter_uncertainties()

GAV_uncertainties = uncertainty.thermo_input_uncertainties
    

In [66]:
# Get all reaction rate rule uncertainties
rate_rule_reactions = [copy.deepcopy(entry.item) for entry in k_entries]
rate_rules = [copy.deepcopy(entry.item) for entry in k_entries]
for i in range(len(rate_rule_reactions)):
    for family in database.kinetics.families:
        try:
            r, p = database.kinetics.families[family].get_labeled_reactants_and_products(
                [sp.molecule[0] for sp in rate_rule_reactions[i].reactants],
                [sp.molecule[0] for sp in rate_rule_reactions[i].products]
            )
            if r and p:
                
                new_rxn = rmgpy.data.kinetics.family.TemplateReaction()
                new_rxn.reactants = r
                new_rxn.products = p
                new_rxn.family = family

                for j in range(len(r)):
                    index1 = get_i_thing(r[j], sp_entries)
                    assert index1 >= 0
                    rate_rule_reactions[i].reactants[j].thermo = entries[index1].data
                for j in range(len(p)):
                    index1 = get_i_thing(p[j], sp_entries)
                    assert index1 >= 0
                    rate_rule_reactions[i].products[j].thermo = entries[index1].data

                template_labels = database.kinetics.families[family].get_reaction_template_labels(new_rxn)
                template = database.kinetics.families[family].retrieve_template(template_labels)
                degeneracy = database.kinetics.families[family].calculate_degeneracy(new_rxn)
                kinetics = database.kinetics.families[family].get_kinetics_for_template(template, degeneracy=degeneracy)[0]

                rate_rule_reactions[i].kinetics = kinetics
                rate_rules[i] = copy.deepcopy(new_rxn)
                rate_rules[i].kinetics = kinetics
                
                
        except (rmgpy.exceptions.UndeterminableKineticsError, rmgpy.exceptions.KineticsError, IndexError, ValueError, rmgpy.exceptions.ActionError):
            pass
        
        if not rate_rule_reactions[i].kinetics:
            try:
                database.kinetics.families[family].add_atom_labels_for_reaction(
                    rate_rule_reactions[i], output_with_resonance=True, save_order=False, relabel_atoms=True
                )
                
                new_rxn = rmgpy.data.kinetics.family.TemplateReaction()
                new_rxn.reactants = [r.molecule[0] for r in rate_rule_reactions[i].reactants]
                new_rxn.products = [p.molecule[0] for p in rate_rule_reactions[i].products]
                new_rxn.family = family
                
                template_labels = database.kinetics.families[family].get_reaction_template_labels(new_rxn)
                template = database.kinetics.families[family].retrieve_template(template_labels)
                degeneracy = database.kinetics.families[family].calculate_degeneracy(new_rxn)
                kinetics = database.kinetics.families[family].get_kinetics_for_template(template, degeneracy=degeneracy)[0]

                rate_rule_reactions[i].kinetics = kinetics
                rate_rules[i] = copy.deepcopy(new_rxn)
                rate_rules[i].kinetics = kinetics
                    
            except (rmgpy.exceptions.UndeterminableKineticsError, rmgpy.exceptions.KineticsError, rmgpy.exceptions.ActionError, IndexError, ValueError):
                pass
            
            
    assert rate_rule_reactions[i].kinetics, f'Failed to get kinetics for {i}'
    assert rate_rules[i].kinetics, f'Failed to get kinetics for {i}'

In [67]:
all_species = []
for i in range(len(rate_rule_reactions)):
    for j in range(len(rate_rule_reactions[i].reactants)):
        try:
            index = all_species.index(rate_rule_reactions[i].reactants[j])
        except ValueError:
            all_species.append(rate_rule_reactions[i].reactants[j])
    for j in range(len(rate_rule_reactions[i].products)):
        try:
            index = all_species.index(rate_rule_reactions[i].products[j])
        except ValueError:
            all_species.append(rate_rule_reactions[i].products[j])

for i in range(len(all_species)):
    all_species[i].thermo.comment = 'Thermo library: fakeLibrary'

In [69]:
uncertainty = rmgpy.tools.uncertainty.Uncertainty(species_list=all_species, reaction_list=rate_rules)
uncertainty.database = database

uncertainty.extract_sources_from_model()
uncertainty.assign_parameter_uncertainties()

GAV_uncertainties = uncertainty.thermo_input_uncertainties
rate_rule_uncertainties = uncertainty.kinetic_input_uncertainties

In [ ]:
rate_rule_reactions[0]

In [ ]:
beyond_uncertainty = []
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
alpha = 0.65
U_DFT = 1.5
DFT_95 = U_DFT * 1.96
plot = False
for i, thermo_entry in enumerate(entries):
    species_index = thermo_entry.index

    U_GAV = GAV_uncertainties[i]
    GAV_95 = U_GAV * 1.96
    
    Ts = np.linspace(300, 1500, 11)
#     Ts = np.linspace(1000, 1100, 11)
    Gs_GAV = np.zeros_like(Ts)
    Gs_DFT = np.zeros_like(Ts)
    for j in range(len(Ts)):
        Gs_GAV[j] = GAV_species[i].get_free_energy(Ts[j]) / 4184  # kcal/mol
        Gs_DFT[j] = thermo_entry.data.get_free_energy(Ts[j]) / 4184  # kcal/mol
        
        
    MAX_DIFF_U_std = np.sqrt(np.float_power(U_DFT, 2.0) + np.float_power(U_GAV, 2.0))
    MAX_DIFF_U_95 = MAX_DIFF_U_std * 1.96  # convert 1 standard deviation of a normal distribution to a 95% interval

    
    if np.any(np.abs(Gs_GAV - Gs_DFT) > MAX_DIFF_U_95):
        beyond_uncertainty.append(species_index)
        
        if plot:
            plt.plot(1000.0 / Ts, Gs_GAV, label='GAV', color=colors[0])
            plt.fill_between(1000.0 / Ts, Gs_GAV, Gs_GAV + GAV_95, alpha=alpha, color=colors[0])
            plt.fill_between(1000.0 / Ts, Gs_GAV, Gs_GAV - GAV_95, alpha=alpha, color=colors[0])


            plt.plot(1000.0 / Ts, Gs_DFT, label='DFT', color=colors[1])
            plt.fill_between(1000.0 / Ts, Gs_DFT, Gs_DFT + DFT_95, alpha=alpha, color=colors[1])
            plt.fill_between(1000.0 / Ts, Gs_DFT, Gs_DFT - DFT_95, alpha=alpha, color=colors[1])

            plt.xlabel('1000 K / T')
            plt.ylabel('Gibbs Energy (kcal / mol)')
            plt.title(f'{species_index}: {thermo_entry.item.smiles} 95% Confidence')
            ax = plt.gca()
            ax.spines['right'].set_visible(False)
            ax.spines['top'].set_visible(False)
            plt.legend()
            plt.show()
        
        
#         print(species_index, thermo_entry)
        
        

    
    
if beyond_uncertainty:
    print(f'You have {len(beyond_uncertainty)} species outside the GAV + DFT 95% confidence interval:')
    for i in beyond_uncertainty:
        print(f'\t{i}')
else:
    print(f'All species have iop(2/9=2000) keyword')
    

### 1.3 Do any of the reaction kinetics differ by more than the rate rule + dlpno-CCSD(T) uncertainty?

## 2. Check parameter values in reasonable range

In [42]:
my_reactions = []
for i, k in enumerate(k_entries):
    
    r_fwd = rmgpy.reaction.Reaction()
    r_fwd.reactants = k_entries[i].item.reactants
    r_fwd.kinetics = k_entries[i].data
    for j in range(len(r_fwd.reactants)):
        index1 = get_i_thing(r_fwd.reactants[j].molecule[0], sp_entries)
        assert index1 >= 0
        
        r_fwd.reactants[j].thermo = entries[index1].data
    
    
    r_fwd.products = k_entries[i].item.products
    for j in range(len(r_fwd.products)):
        index1 = get_i_thing(r_fwd.products[j].molecule[0], sp_entries)
        assert index1 >= 0
        
        r_fwd.products[j].thermo = entries[index1].data
    
    r_rev = get_reverse_reaction(r_fwd)
    
    
    my_reactions.append([r_fwd, r_rev])

### 2.1 Negative reaction barriers?

In [43]:
negative_reactions = []
for r_fwd, r_rev in my_reactions:
    if r_fwd.kinetics.Ea.value_si <= 1.0 or r_rev.kinetics.Ea.value_si <= 1.0:
        negative_reactions.append(database_fun.get_unique_reaction_index(r_fwd))

if negative_reactions:
    print(f'You have {len(negative_reactions)} negative reaction barriers:')
    for i in negative_reactions:
        print(f'\t{i}')
else:
    print('All reaction barriers >= 0')

You have 14 negative reaction barriers:
	288
	404
	804
	808
	1288
	4721
	4728
	4729
	4778
	4779
	4796
	9358
	10105
	10106


### 2.2 $|n| \geq 4.0$ ?

In [44]:
large_n = []
N_THRESHOLD = 4.0
for r_fwd, r_rev in my_reactions:
    if np.abs(r_fwd.kinetics.n.value_si) >= N_THRESHOLD or np.abs(r_rev.kinetics.n.value_si) >= N_THRESHOLD:
        large_n.append(database_fun.get_unique_reaction_index(r_fwd))

if large_n:
    print(f'You have {len(large_n)} reactions with n >= {N_THRESHOLD}:')
    for i in large_n:
        print(f'\t{i}')
else:
    print(f'All reaction kinetics have |n| <= {N_THRESHOLD}')

You have 22 reactions with n >= 4.0:
	253
	280
	286
	321
	419
	714
	808
	1111
	1287
	1288
	4728
	4729
	4736
	4778
	4779
	4796
	5046
	9060
	10105
	10106
	10111
	10168


### 2.3 Any of the parameters have ridiculous exponents?

In [45]:
UPPER_LIMIT_A = 1e18
large_exp_A = []
for r_fwd, r_rev in my_reactions:
    if np.abs(r_fwd.kinetics.A.value_si) >= UPPER_LIMIT_A or np.abs(r_rev.kinetics.A.value_si) >= UPPER_LIMIT_A:
        large_exp_A.append(database_fun.get_unique_reaction_index(r_fwd))

if large_exp_A:
    print(f'You have {len(large_exp_A)} reactions with A >= {UPPER_LIMIT_A}:')
    for i in large_exp_A:
        print(f'\t{i}')
else:
    print(f'All reaction kinetics have |A| <= {UPPER_LIMIT_A}')
    
UPPER_LIMIT_EA = 3e6  # J/mol
large_exp = []
for r_fwd, r_rev in my_reactions:
    if np.abs(r_fwd.kinetics.Ea.value_si) >= UPPER_LIMIT_EA or np.abs(r_rev.kinetics.Ea.value_si) >= UPPER_LIMIT_EA:
        large_exp.append(database_fun.get_unique_reaction_index(r_fwd))

if large_exp:
    print(f'You have {len(large_exp)} reactions with A >= {UPPER_LIMIT_EA}:')
    for i in large_exp:
        print(f'\t{i}')
else:
    print(f'All reaction kinetics have |Ea| <= {UPPER_LIMIT_EA}')

All reaction kinetics have |A| <= 1e+18
All reaction kinetics have |Ea| <= 3000000.0


### 2.4 Collision rate violators?

## 3. Errors from Software Components

### 3.1 Any final ts/conformer Gaussian files with 14+ atoms missing the iop(2/9=2000) keyword?

In [46]:
missing_iop_keyword = []
missing_iop_keyword_and_large = []
for thermo_entry in entries:
    species_index = thermo_entry.index
    gaussian_log = glob.glob(os.path.join(DFT_DIR, 'thermo', f'species_{species_index:04}', 'arkane', 'conformer_*.log'))[0]
    found_iop_keyword = False
    atom_count = len(thermo_entry.item.atoms)
    with open(gaussian_log, 'r') as f:
        for i in range(200):
            line = f.readline()
            if '2/9=2000' in line:
                found_iop_keyword = True
                break
    if not found_iop_keyword:
        missing_iop_keyword.append(species_index)
        if atom_count >= 14:
            missing_iop_keyword_and_large.append(species_index)

if missing_iop_keyword_and_large:
    print(f'You have {len(missing_iop_keyword_and_large)} species with more than 14 atoms and no iop(2/9=2000) keyword:')
    for i in missing_iop_keyword_and_large:
        print(f'\t{i}')
else:
    print(f'All species have iop(2/9=2000) keyword')
    
    
    
missing_iop_keyword = []
missing_iop_keyword_and_large = []
for kinetics_entry in k_entries:
    reaction_index = kinetics_entry.index
    gaussian_log = glob.glob(os.path.join(DFT_DIR, 'kinetics', f'reaction_{reaction_index:06}', 'arkane', 'ts', 'fwd_*.log'))[0]
    found_iop_keyword = False
    atom_count = np.sum([len(r.molecule[0].atoms) for r in kinetics_entry.item.reactants])
    with open(gaussian_log, 'r') as f:
        for i in range(1000):
            line = f.readline()
            if '2/9=2000' in line:
                found_iop_keyword = True
                break
    if not found_iop_keyword:
        missing_iop_keyword.append(reaction_index)
        if atom_count >= 14:
            missing_iop_keyword_and_large.append(reaction_index)

if missing_iop_keyword_and_large:
    print(f'You have {len(missing_iop_keyword_and_large)} reactions with more than 14 atoms and no iop(2/9=2000) keyword:')
    for i in missing_iop_keyword_and_large:
        print(f'\t{i}')
else:
    print(f'All reactions have iop(2/9=2000) keyword')

All species have iop(2/9=2000) keyword
You have 7 reactions with more than 14 atoms and no iop(2/9=2000) keyword:
	278
	286
	289
	314
	324
	4721
	4778


### 3.2 Did Arkane complain about any of the rotor energies being less than the lowest energy conformer?

## 4. Other

### 4.1 Flying Hydrogen atoms? (incorrect atom numbering on hindered rotor scans)